In [1]:
%pip install anthropic dotenv



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from anthropic import Anthropic
from dotenv import load_dotenv
from datetime import datetime,timedelta
from anthropic.types import ToolParam, Message
import json

In [3]:
#load_dotenv()
client = Anthropic()


In [4]:

# message = client.messages.create(
#     model="claude-sonnet-5",
#     max_tokens=1024,
#     messages=[
#         {"role": "user", "content": "Explain quantum computing in one sentence."}
#     ]
# )

In [17]:
def add_user_message(messages,message):
    user_message = {
        "role":"user",
        "content": message.content if isinstance(message, Message) else message,
        }
    messages.append(user_message)

def add_assistant_message(messages,message):
    assistant_message = {
        "role":"assistant",
        "content": message.content if isinstance(message, Message) else message,
        }
    messages.append(assistant_message)
def chat(messages,system=None,stop_sequences=None,tools=None):
    params = {
        "model" :"claude-sonnet-5",
        "max_tokens":1000,
        "messages": messages,
    }

    if stop_sequences:
        params["stop_sequences"] = stop_sequences

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system
    message = client.messages.create(**params)
    return message

def text_from_message(message):
     return "\n".join([block.text for block in message.content if block.type == "text"])

In [6]:
#chat
# messages =[]
# while True:
#     user_input = input("> ")
#     print(">",user_input)

#     add_user_message(messages,user_input)
#     answer = chat(messages)
#     add_assistant_message(messages,answer)
#     print("----")
#     print(answer)
#     print("----")


In [ ]:
# messages = []
# add_user_message(messages,
#                  "Write a python function to check a string for duplicate characters")
# answer = chat(messages,system="You are a python engineer who writes very concise programs")

In [34]:
def get_current_datetime(date_format = "%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

In [ ]:
# messages = []
# messages.append(
#     {
#         "role": "user",
#         "content": "what is the exact time, formatted as HH:MM:SS?Also, please provide the current date in YYYY-MM-DD format.",
#     } 
# )
# chat(messages,tools=[get_current_datetime_schema])
# response = client.messages.create(
#     model = "claude-haiku-5-5",
#     max_tokens = 1000,
#     messages= messages,
#     tools = [get_current_datetime_schema],
# )
# messages.append({
#     "role": "assistant",
#     "content": response.content
#     })
# messages

Message(id='msg_011CfrrmmbNoxZpgarqSgZtp', container=None, content=[ThinkingBlock(signature='EoMFCsQCCBIYAipA/X6oyM0zH5iv2PU5VPxxOEFMR4pWoFlrEFHeHcC76Q2mf8SSifriTktDfrhekCCZ1OQ30T3SKlGqUvoZsbN/7jIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRhNWM1OTNkNS1iNWVhLTQzZGYtYTBmZi0xYmE3YTVlMmY5YjKaAREKD2NsYXVkZS1zb25uZXQtNagBy4Gk1gbCAToIBBAAGgwCwPADTzcF2WMEJq0gATIEEnHPJToEODMjikIEJaN750oEmkzKpFABWgTZBeFWYAJoAXAA2gFgChZwcmVmaXhfYmVmb3JlX2Jsb2NrX3YxEiCB3bPfBLKeQuFwAtn73ax1FQlo87W52vcrS5STQqDfOSIAMiDCwBPIapyS4i+Wbz2GF3aPi3pEyQKtzAl7+h4JivtOH0gDEgxSc1RzG/yg5vIcpCoaDHdneTpFuDTCVRsiDyIwb1/pRgFeA1TiQ4oD028h1ZHZUpjXY/3FB7wFE2sXi2g0cdEbKpAlkQLWnYua9JGBKusB/NGxNSmwHjRzRMC+ajAu5MPeyiFfc5+Siwp5z86hqkirVxIAdaJlvpXi0CwFE3MlNv8dxEjw9YfByHmfY2PT3dl3oRqg1mbIz3xUZ2jdDAnUqF02lHP1eimmVcI0iCnHJpr9OqkkDH7bMx20/WzXT0Zr6jzgM6xOPWkbU+S2Kr8+r/U3ByHpKKeNyBLFFaXCnUIXQLcpIgJNdU3d4xjq0otG2b/KIO9+6IH3vz0wleECv7qrzt/s99W0gheV/mKt6LjkoVfjzktpFBpcbOj2yFud21Um8BhB6vwenuphGgbOEoFKGPZcTs2RnhgB', thinking='', type='thinking'), ToolUseBl

In [ ]:
# result = get_current_datetime(**response.content[0].input)

In [5]:
# messages.append({
#     "role": "user",
#     "content": [
#         {
#             "type": "tool_result",
#             "tool_use_id": response.content[0].id,
#             "content": result,
#             "is_error": False
#         }
#     ]
    
# })
# messages

In [6]:
# client.messages.create(
#     model = "claude-haiku-5-5",
#     max_tokens = 1000,
#     messages=messages,
#     tools = [get_current_datetime_schema]
# )

In [35]:
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    raise ValueError(f"Unsupported tool: {tool_name}")

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type =="tool_use"
    ]
    tool_result_blocks = []
    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_block = {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": json.dumps(tool_output),
            "is_error": False
            }
        except Exception as e:
            tool_result_block = {
            "type": "tool_result",
            "tool_use_id": tool_request.id,
            "content": f"Error: {e}",
            "is_error": True
            }
        tool_result_blocks.append(tool_result_block)
    return tool_result_blocks
        

In [36]:
def run_conversation(messages):
    while True:
        response = chat(
            messages, 
            tools = [get_current_datetime_schema,
                      add_duration_to_datetime_schema,
                      set_reminder_schema,
                      ],
        )

        add_assistant_message(messages,response)
        print(text_from_message(response))

        if response.stop_reason !="tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages,tool_results)
    return messages

In [37]:
messages =[]
add_user_message(
    messages,
    "Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050."
)
run_conversation(messages)


Based on the calculation, January 1, 2050 + 177 days falls on **June 27, 2050**. I'll set a reminder for your doctor's appointment on that date (defaulting to 9:00 AM since no specific time was given).
----
Setting the following reminder for 2050-06-27T09:00:00:
Doctor's appointment
----
Your reminder for the **doctor's appointment** has been set for **June 27, 2050 at 9:00 AM** (177 days after January 1, 2050). Let me know if you'd like to adjust the time!


[{'role': 'user',
  'content': 'Set a reminder for my doctors appointment. Its 177 days after Jan 1st, 2050.'},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='EogECsQCCBIYAipAXXX4AVP9lJc7YtDkYBn5ppHOTYJkhN3N4lQcDVBOvX8rZw7EnF68CQDx10U6Ts9PAphTqvWnZ+eSum6rhW9F+TIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRhNWM1OTNkNS1iNWVhLTQzZGYtYTBmZi0xYmE3YTVlMmY5YjKaAREKD2NsYXVkZS1zb25uZXQtNagBwqSk1gbCAToIBBAAGgwCoD4DTzcFH/wEJq0gATIEEnHPJToEODMjikIEt9YqdUoEti143VADWgTZBeFWYAJoAXAA2gFgChZwcmVmaXhfYmVmb3JlX2Jsb2NrX3YxEiBk518UBrtqLIyndqyvGUmaKrVL0PQeITrXpHL5Cl8ZQCIAMiDCwBPIapyS4i+Wbz2GF3aPi3pEyQKtzAl7+h4JivtOH0gDEgwOKq/CJPotWy8pi+gaDAKIw72EjUBornlDfiIwjGPO/9VNzBTo0Ui8KJQdDTV1rTXHQCvnVZTjT9qrqHEj8h5EDSF/YFVS+jKQrgNHKnFiT6qbh/PVEyr/w7bbZf9B5bC0uHQA0Vcd3iMy6qyytxfUpBQoJW/yGZgfWVXRP4Wu/wOLTYfky3RXMerC2yVX6krxvd1zToMLm59XGASTuRe4O+urh4t4fKHpB+EnNiSleyEl9jzw76/Nz6iurbydZBgB', thinking='', type='thinking'),
   ToolUseBlock(id='toolu_019ANDLFhNgmB1GFimHinqJ2', caller=DirectCaller(type='direct'), inp

In [26]:
messages

[{'role': 'user',
  'content': 'What is the current time in HH:MM format? Also, what is the current time in SS format?'},
 {'role': 'assistant',
  'content': [ThinkingBlock(signature='EpUECsQCCBIYAipAdUHOSMc6oje8SGOpStMlhlXAIi6yFY7cy3PHPJPhhwuP8iaJquFfPmEY3zp6NOpP7ZYapctiHUzBZHN0TFf3FDIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiRhNWM1OTNkNS1iNWVhLTQzZGYtYTBmZi0xYmE3YTVlMmY5YjKaAREKD2NsYXVkZS1zb25uZXQtNagBxf6j1gbCAToIBBAAGgwCwPADTzcFQRQEJq0gATIEEnHPJToEODMjikIEJaN750oEmkzKpFABWgTZBeFWYAJoAXAA2gFgChZwcmVmaXhfYmVmb3JlX2Jsb2NrX3YxEiBy75rNvpHkwMtALmzImUNan/Vas/IAgAmgn/ZzsrVk9SIAMiDCwBPIapyS4i+Wbz2GF3aPi3pEyQKtzAl7+h4JivtOH0gDEgxFSQ6outjiXMTKY7oaDLq4VhbPwLNFfC6u1yIwa7oBSdOsM2GNkoYRKyh2A4S/OJG0XkMkA/aPR42oQxlD7IpXK9NQdo3P4e1jow8aKn7iId45aLDgPhfQ13Eu3VH8u4f7OGs0hxjYBOaW4I4D8l+x+bBxOwRKHfNVA2vV8PRtSkgzFCCsztbr27usA+xPwhD0gV2mhS2CMdJMnmWpxI9uLI4H4Oq6x8IfvYHRyNXwRx5ks1BnlaOcosZb6IQYJO0SjfMZzyNSLh6dM3AYAQ==', thinking='', type='thinking'),
   ToolUseBlock(id='toolu_01P4wFZfWnFVspn9y6dXCUQn', caller=Di